## Modelo para vincular personas no vinculadas por el modelo general

Se parte de las predicciones obtenidas en el modelo general. Se conservan aquellas personas de censo que no fueron vinculadas a personas de registro con probabilidad mayor o igual a 0.2. Además, se descartan algunas personas consideradas no vinculables por falta de datos. Por su parte, se utiliza la lista completa de personas del hub_personas_DNIC, independientemente de si fueron vinculadas en la etapa anterior o no.

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import splink

%matplotlib inline

os.chdir(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraa')

df_l = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_l_censo_sin_snsa.parquet')
df_r = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\df_r_rraa_sin_snsa.parquet')
censo = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\data\raw\censo.parquet')

predictions_max_match_prob = pd.read_parquet(r'C:\Users\lpescetto\Desktop\vinculacion_censo_rraaa\output\pred_modelo4.parquet')

no_vinculados = pd.merge(df_l.rename(columns = {"unique_id":"unique_id_l"}), predictions_max_match_prob["unique_id_l"], how = "left", indicator = True)

no_vinculados = no_vinculados[no_vinculados["_merge"] == "left_only"].merge(censo[["id_censo", "tipo_caso", "docextnro"]].rename(columns = {"id_censo":"unique_id_l", "docextnro":"documento_extranjero"}), how = "left").drop(columns="_merge")

documentos = df_l.rename(columns={"documento":"documento_l"})["documento_l"].value_counts()

no_vinculados["doc_unico"] = no_vinculados["documento"].apply(lambda x: 1 if pd.notna(x) and documentos.get(x, 0) == 1 else 0)

no1 = pd.isnull(no_vinculados["documento"]) & pd.isnull(no_vinculados["fecha_nacimiento"]) & pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])
#no2 = (no_vinculados["doc_unico"] == 0) & ((pd.isnull(no_vinculados["fecha_nacimiento"]) & (pd.isnull(no_vinculados["primer_nombre"]) | pd.isnull(no_vinculados["primer_apellido"]))) | (pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])))
no3 = (no_vinculados["doc_unico"] == 0) & ((pd.isnull(no_vinculados["primer_nombre"]) | pd.isnull(no_vinculados["primer_apellido"])))
#no3 = (no_vinculados["doc_unico"] == 0) & pd.isnull(no_vinculados["fecha_nacimiento"]) & pd.isnull(no_vinculados["primer_nombre"]) & pd.isnull(no_vinculados["primer_apellido"])

#& pd.isnull(no_vinculados["segundo_nombre"]) & pd.isnull(no_vinculados["segundo_apellido"])

vinculables = no_vinculados[~(no1 | no3)]

Se tiene un total de 66.051 personas para este modelo. Pero se entrena con todas las observaciones.

In [5]:
vinculables.columns

Index(['unique_id_l', 'primer_nombre', 'segundo_nombre', 'primer_apellido',
       'segundo_apellido', 'documento', 'documento_valido', 'fecha_nacimiento',
       'ano_nacimiento', 'mes_nacimiento', 'dia_nacimiento', 'id_sexo',
       'id_departamento', 'nombres', 'apellidos', 'ano_nacimiento_imputados',
       'id_departamento_ajustado', 'tipo_caso', 'documento_extranjero',
       'doc_unico'],
      dtype='object')

In [4]:
df_r.columns

Index(['unique_id', 'primer_nombre', 'segundo_nombre', 'primer_apellido',
       'segundo_apellido', 'documento', 'documento_valido', 'fecha_nacimiento',
       'ano_nacimiento', 'mes_nacimiento', 'dia_nacimiento', 'id_sexo',
       'id_departamento', 'nombres', 'apellidos', 'id_departamento_ajustado',
       'ano_nacimiento_imputados'],
      dtype='object')

In [8]:
df_l.columns

Index(['unique_id', 'primer_nombre', 'segundo_nombre', 'primer_apellido',
       'segundo_apellido', 'documento', 'documento_valido', 'fecha_nacimiento',
       'ano_nacimiento', 'mes_nacimiento', 'dia_nacimiento', 'id_sexo',
       'id_departamento', 'nombres', 'apellidos', 'ano_nacimiento_imputados',
       'id_departamento_ajustado'],
      dtype='object')

In [6]:
df_l.columns == df_r.columns

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False, False])

In [36]:
#df_l = vinculables.drop(columns=["tipo_caso", "documento_extranjero", "doc_unico"]).rename(columns = {"unique_id_l":"unique_id"}).reindex(columns=df_r.columns)

# cambio de lugar las últimas 2 columnas
# cols = df_l.columns.tolist()
# cols[-1], cols[-2] = cols[-2], cols[-1]
# df_l = df_l[cols]

df_l = df_l.reindex(columns=df_r.columns)

df_l["unique_id"] = df_l["unique_id"] + max(df_r["unique_id"])

# se chequea que los df cumplan los requisitos para implementar el modelo

# 1. id unico en cada tabla
print("1:")
print("id_unico en censo:", len(df_l) == len(df_l.drop_duplicates(subset = "unique_id")))
print("id_unico en rraa:", len(df_r) == len(df_r.drop_duplicates(subset = "unique_id")))
print("\n" * 1)

# 2. los tipos de datos deben ser iguales
print("2:")
print(df_l.dtypes[1:] == df_r.dtypes[1:])

# 3. se recuerda que los nulls son tratados de forma distinto que las strings vacías (en los nombres hay strings vacías y en el resto de las variables nulls)

1:
id_unico en censo: True
id_unico en rraa: True


2:
primer_nombre               True
segundo_nombre              True
primer_apellido             True
segundo_apellido            True
documento                   True
documento_valido            True
fecha_nacimiento            True
ano_nacimiento              True
mes_nacimiento              True
dia_nacimiento              True
id_sexo                     True
id_departamento             True
nombres                     True
apellidos                   True
id_departamento_ajustado    True
ano_nacimiento_imputados    True
dtype: bool


In [10]:
from splink.duckdb.linker import DuckDBLinker
linker_censo = DuckDBLinker(df_l)
linker_rraa = DuckDBLinker(df_r)

linker_censo.missingness_chart()

alt.LayerChart(...)

In [5]:
from splink.duckdb.linker import DuckDBLinker
from splink.duckdb.blocking_rule_library import block_on

settings = {"link_type": "link_only"}

# para estimar la cantidad de comparaciones, uso el df con el que voy a predecir
linker_br = DuckDBLinker([df_r, vinculables.drop(columns=["tipo_caso", "documento_extranjero", "doc_unico"]).rename(columns = {"unique_id_l":"unique_id"}).reindex(columns=df_r.columns)], settings)

br1 = block_on(["primer_nombre", "primer_apellido"])

#br2 = block_on(["ano_nacimiento", "id_sexo"]) (1,755,398,121)
#br3 = block_on(["ano_nacimiento_imputados", "id_sexo"]) (181,560,669)
br9 = block_on(["fecha_nacimiento", "id_sexo"])

#br4 = block_on(["ano_nacimiento", "id_departamento_ajustado"]) (1,100,713,856)
br5 = block_on(["ano_nacimiento_imputados", "id_departamento_ajustado"]) #(89,715,284)
br8 = block_on(["fecha_nacimiento", "id_departamento_ajustado"])

#br6 = block_on(["id_departamento_ajustado", "substr(primer_nombre, 1,1)", "substr(primer_apellido, 1,1)"]) (488,828,046)
br7 = block_on(["id_departamento_ajustado", "substr(primer_nombre, 1,2)", "substr(primer_apellido, 1,2)"])

list_br = [br1, br7, br8, br9]

# for br in list_br:
#     count = linker_br.count_num_comparisons_from_blocking_rule(br)
#     print(f"Comparisons generated by '{br.blocking_rule_sql}': {count:,.0f}")

In [28]:
blocking_rules = [br1, br9, br5, br8, br7]

linker_br.cumulative_num_comparisons_from_blocking_rules_chart(blocking_rules)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

alt.Chart(...)

In [61]:
def get_comparacion_nombres_especial(nombre_comparacion, var1, var2, var_comb):

    comparacion = {
        "output_column_name": f"{nombre_comparacion}",
        "comparison_description": "no description",
        "comparison_levels": [
            {
                "sql_condition": f"({var1}_l IS NULL AND {var2}_l IS NULL) OR ({var1}_r IS NULL AND {var2}_r IS NULL)",
                "label_for_charts": "Null",
                "is_null_level": True,
            },
            {
                "sql_condition": f"jaro_winkler_similarity({var1}_l, {var1}_r) > 0.88 AND jaro_winkler_similarity({var2}_l, {var2}_r) > 0.88",
                "label_for_charts": "Exact match both"#,
                #"tf_adjustment_column": f"{var_comb}",
                #"tf_adjustment_weight": 1.0,
                #"tf_minimum_u_value": 0.001,
            },
            {
                "sql_condition": f"jaro_winkler_similarity({var1}_l, {var2}_r) > 0.88 AND jaro_winkler_similarity({var2}_l, {var1}_r) > 0.88",
                "label_for_charts": "Exact match both swapped"
            },
            {
                "sql_condition": f"(jaro_winkler_similarity({var1}_l, {var2}_r) > 0.8 AND {var2}_l IS NULL) OR (jaro_winkler_similarity({var1}_l, {var1}_r) > 0.88)",
                "label_for_charts": "rraa2 - censo1"
            },
            # {
            #     "sql_condition": f"jaro_winkler_similarity({var1}_l, {var1}_r) > 0.88",
            #     "label_for_charts": "Exact match first"#,
            #     # "tf_adjustment_column": f"{var1}",
            #     # "tf_adjustment_weight": 1.0,
            #     # "tf_minimum_u_value": 0.001,
            # },

            {"sql_condition": "ELSE", "label_for_charts": "All other comparisons"},
        ],
    }

    return comparacion

In [62]:
from scripts.comparaciones import get_comparacion_fecha_nacimiento, get_comparacion_ano_nacimiento_imputados, get_comparacion_nombres_combinados, get_comparacion_nombre
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_template_library as ctl
import splink.duckdb.comparison_level_library as cll

comparacion_fecha_nacimiento = get_comparacion_fecha_nacimiento(con_dc = True)
comparacion_ano_nacimiento_imputados = get_comparacion_ano_nacimiento_imputados()

comparacion_sexo = cl.exact_match("id_sexo")
#comparacion_documento = cl.damerau_levenshtein_at_thresholds("documento", [1,2])
#comparacion_documento_extranjero = cl.levenshtein_at_thresholds("documento_extranjero", [1])
#comparacion_id_pais_documento = cl.exact_match("id_pais_documento")
#comparacion_id_tipo_documento = cl.exact_match("id_tipo_documento")

#comparacion_departamento = cl.exact_match("id_departamento", term_frequency_adjustments = True)
comparacion_departamento_ajustado = cl.exact_match("id_departamento_ajustado", term_frequency_adjustments = True)

#comparacion_nombres = get_comparacion_nombres_combinados("primer_nombre", "segundo_nombre", "nombres")
#comparacion_apellidos = get_comparacion_nombres_combinados("primer_apellido", "segundo_apellido", "apellidos")

comparacion_primer_nombre = get_comparacion_nombre("primer_nombre", usar_tfa = True)
comparacion_segundo_nombre = get_comparacion_nombre("segundo_nombre", usar_tfa = True)
comparacion_primer_apellido = get_comparacion_nombre("primer_apellido", usar_tfa = True)
comparacion_segundo_apellido = get_comparacion_nombre("segundo_apellido", usar_tfa = True)

comparacion_nombres_combinados = get_comparacion_nombres_especial("comparacion_nombres", "primer_nombre", "segundo_nombre", "nombres")

In [63]:
settings = {
    "link_type": "link_only",
    "comparisons": [
        comparacion_fecha_nacimiento,
        comparacion_ano_nacimiento_imputados,
        #comparacion_primer_nombre,
        #comparacion_segundo_nombre,
        comparacion_nombres_combinados,
        comparacion_primer_apellido,
        comparacion_segundo_apellido,
        comparacion_sexo,
        comparacion_departamento_ajustado
    ],
    "blocking_rules_to_generate_predictions": [
        br1, br9, br5, br8, br7
    ],
    "em_convergence": 1e-8,
    "max_iterations": 25,
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = DuckDBLinker([df_l, df_r], settings)

# Estimación de los parámetros

## 1. Probabilidad de que dos individuos sean match

In [64]:
deterministic_rules = [
    "l.documento = r.documento"
    ]

linker.estimate_probability_two_random_records_match(deterministic_rules, recall = 0.90)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Probability two random records match is estimated to be  1.65e-07.
This means that amongst all possible pairwise record comparisons, one in 6,049,405.82 are expected to match.  With 19,209,578,998,385 total possible comparisons, we expect a total of around 3,175,448.89 matching pairs


## 2. Probabilidad de vincular dos observaciones dado que NO son match (u)

In [65]:
linker.estimate_u_using_random_sampling(max_pairs=1e8)

----- Estimating u probabilities using random sampling -----


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - fecha_nacimiento (no m values are trained).
    - ano_nacimiento_imputados (no m values are trained).
    - comparacion_nombres (no m values are trained).
    - primer_apellido (no m values are trained).
    - segundo_apellido (no m values are trained).
    - id_sexo (no m values are trained).
    - id_departamento_ajustado (no m values are trained).


## 3. Probabilidad de vincular dos observaciones dado que son match (m)

In [66]:
linker.debug_mode = False

t_br1 = block_on(["documento"])
training_session1 = linker.estimate_parameters_using_expectation_maximisation(t_br1)


----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
l."documento" = r."documento"

Parameter estimates will be made for the following comparison(s):
    - fecha_nacimiento
    - ano_nacimiento_imputados
    - comparacion_nombres
    - primer_apellido
    - segundo_apellido
    - id_sexo
    - id_departamento_ajustado

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 1: Largest change in params was 0.858 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

Iteration 6: Largest change in params was 9.44e-07 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 7: Largest change in params was 8.55e-08 in probability_two_random_records_match


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Iteration 8: Largest change in params was 7.74e-09 in probability_two_random_records_match

EM converged after 8 iterations

Your model is fully trained. All comparisons have at least one estimate for their m and u values


In [68]:
linker.match_weights_chart()

alt.VConcatChart(...)

In [60]:
linker.m_u_parameters_chart()

alt.HConcatChart(...)

In [41]:
df_predictions = linker.find_matches_to_new_records(vinculables.drop(columns=["tipo_caso", "documento_extranjero", "doc_unico"]).rename(columns = {"unique_id_l":"unique_id"}).reindex(columns=df_r.columns),
                                                    blocking_rules = [br1, br9, br5, br8, br7], match_weight_threshold = -2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [50]:
prueba = vinculables.drop(columns=["tipo_caso", "documento_extranjero", "doc_unico"]).rename(columns = {"unique_id_l":"unique_id"}).reindex(columns=df_r.columns).iloc[0:1, :]

df_predictions_prueba = linker.find_matches_to_new_records(prueba, match_weight_threshold= -4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [52]:
predictions_prueba = df_predictions_prueba.as_pandas_dataframe()

In [42]:
predictions = df_predictions.as_pandas_dataframe()

predictions = predictions["u"]

In [21]:
df_predictions = linker.predict(threshold_match_probability=0.2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
predictions = df_predictions.as_pandas_dataframe()

In [31]:
sum(df_r["unique_id"].isin(df_l["unique_id"]))

77166